# 01 - Data Exploration & Quality Control

This notebook explores the downloaded STAGES + MESA data after validation.

**Steps:**
1. Load and summarize the validation quality report
2. Visualize inclusion/exclusion statistics
3. Explore sample actigraphy traces overlaid with PSG hypnograms
4. Analyze sleep stage label distributions (class imbalance)
5. Cross-site distribution analysis for STAGES

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from lxml import etree

sns.set_theme(style="whitegrid")
%matplotlib inline

# Project paths - update these when running on Kodiak
PROJECT_DIR = Path("/data/priorj/sleep_stage_classification")
DATASETS_DIR = PROJECT_DIR / "datasets"
LOGS_DIR = PROJECT_DIR / "logs"

# For local development, use relative paths:
# PROJECT_DIR = Path("..").resolve()
# DATASETS_DIR = PROJECT_DIR / "datasets"
# LOGS_DIR = PROJECT_DIR / "logs"

## 1. Load Validation Report

In [ ]:
# Load the quality report from validate_data.py
report_path = LOGS_DIR / "data_quality_report.csv"
df = pd.read_csv(report_path)
print(f"Total subjects in report: {len(df)}")
print(f"\nDataset breakdown:")
print(df["dataset"].value_counts())
print(f"\nInclusion breakdown:")
print(df["include"].value_counts())
df.head(10)

## 2. Inclusion/Exclusion Statistics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Overall inclusion
inc_counts = df.groupby(["dataset", "include"]).size().unstack(fill_value=0)
inc_counts.plot(kind="bar", ax=axes[0], color=["#e74c3c", "#2ecc71"])
axes[0].set_title("Subject Inclusion by Dataset")
axes[0].set_ylabel("Number of subjects")
axes[0].legend(["Excluded", "Included"])
axes[0].tick_params(axis="x", rotation=0)

# Exclusion reasons
excluded = df[df["include"] == False]
if len(excluded) > 0:
    reason_counts = excluded["exclusion_reason"].value_counts().head(10)
    reason_counts.plot(kind="barh", ax=axes[1], color="#e74c3c")
    axes[1].set_title("Top Exclusion Reasons")
    axes[1].set_xlabel("Count")

# Signal availability
included = df[df["include"] == True]
signal_cols = ["has_pleth", "has_spo2", "has_hr"]
signal_avail = included[signal_cols].mean() * 100
signal_avail.plot(kind="bar", ax=axes[2], color="#3498db")
axes[2].set_title("Signal Availability (Included Subjects)")
axes[2].set_ylabel("% of subjects")
axes[2].set_ylim(0, 105)
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Quality score distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i, dataset in enumerate(["mesa", "stages"]):
    subset = df[df["dataset"] == dataset]
    axes[i].hist(subset["quality_score"], bins=20, color="#3498db", edgecolor="white")
    axes[i].axvline(x=0.6, color="red", linestyle="--", label="Inclusion threshold")
    axes[i].set_title(f"{dataset.upper()} Quality Score Distribution")
    axes[i].set_xlabel("Quality Score")
    axes[i].set_ylabel("Count")
    axes[i].legend()

plt.tight_layout()
plt.show()

## 3. Sample Actigraphy + Hypnogram Visualization

Overlay actigraphy activity counts with PSG sleep stage hypnograms for a few good subjects.

In [ ]:
def load_hypnogram(annot_path):
    """Load sleep stages from annotation XML as (epoch_index, stage) pairs."""
    tree = etree.parse(str(annot_path))
    root = tree.getroot()
    
    stage_map = {
        "Wake": 0, "Stage 1": 1, "Stage 2": 2,
        "Stage 3": 3, "Stage 4": 3, "REM": 5
    }
    
    epochs = {}
    for event in root.iter():
        tag = event.tag.split("}")[-1] if "}" in event.tag else event.tag
        if tag != "ScoredEvent":
            continue
        
        data = {}
        for child in event:
            ctag = child.tag.split("}")[-1] if "}" in child.tag else child.tag
            data[ctag] = child.text
        
        if "Stages" not in data.get("EventType", ""):
            continue
        
        concept = data.get("EventConcept", "")
        start = float(data.get("Start", 0))
        duration = float(data.get("Duration", 30))
        
        stage = None
        for key, val in stage_map.items():
            if key in concept:
                stage = val
                break
        if stage is None:
            continue
        
        epoch_start = int(start / 30)
        n_epochs = max(1, int(duration / 30))
        for e in range(epoch_start, epoch_start + n_epochs):
            epochs[e] = stage
    
    if not epochs:
        return np.array([]), np.array([])
    
    max_epoch = max(epochs.keys()) + 1
    times = np.arange(max_epoch) * 30 / 3600  # hours
    stages = np.array([epochs.get(i, np.nan) for i in range(max_epoch)])
    return times, stages

print("Hypnogram loader ready.")

In [ ]:
# Plot actigraphy + hypnogram for sample subjects
sample_subjects = included.head(5)
stage_names = {0: "Wake", 1: "N1", 2: "N2", 3: "N3", 5: "REM"}
stage_colors = {0: "#e74c3c", 1: "#f39c12", 2: "#3498db", 3: "#2c3e50", 5: "#2ecc71"}

for _, row in sample_subjects.iterrows():
    subj_id = row["subject_id"]
    dataset = row["dataset"]
    ds_dir = DATASETS_DIR / dataset
    
    # Load actigraphy
    actig_files = list((ds_dir / "actigraphy").glob(f"*{subj_id}*.csv"))
    annot_files = list((ds_dir / "polysomnography" / "annotations-events-nsrr").glob(f"*{subj_id}*.xml"))
    
    if not actig_files or not annot_files:
        continue
    
    actig_df = pd.read_csv(actig_files[0])
    act_col = next((c for c in ["activity", "Activity"] if c in actig_df.columns), None)
    if act_col is None:
        continue
    
    epoch_sec = 60 if dataset == "stages" else 30
    actig_hours = np.arange(len(actig_df)) * epoch_sec / 3600
    activity = actig_df[act_col].values
    
    # Load hypnogram
    hyp_times, hyp_stages = load_hypnogram(annot_files[0])
    
    # Plot
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
    
    ax1.plot(actig_hours, activity, color="#3498db", alpha=0.7, linewidth=0.5)
    ax1.set_ylabel("Activity Count")
    ax1.set_title(f"{dataset.upper()} - {subj_id}: Actigraphy + Hypnogram")
    
    if len(hyp_times) > 0:
        for stage_val, stage_name in stage_names.items():
            mask = hyp_stages == stage_val
            if np.any(mask):
                ax2.scatter(hyp_times[mask], hyp_stages[mask], 
                          c=stage_colors[stage_val], s=2, label=stage_name)
        ax2.set_yticks(list(stage_names.keys()))
        ax2.set_yticklabels(list(stage_names.values()))
        ax2.invert_yaxis()
        ax2.legend(loc="upper right", markerscale=5)
    
    ax2.set_xlabel("Time (hours)")
    ax2.set_ylabel("Sleep Stage")
    plt.tight_layout()
    plt.show()

## 4. Label Distribution Analysis (Class Imbalance)

In [ ]:
# Count stage labels across all included subjects
stage_counts = {"WAKE": 0, "N1": 0, "N2": 0, "N3": 0, "REM": 0}
stage_map_inv = {0: "WAKE", 1: "N1", 2: "N2", 3: "N3", 5: "REM"}

per_dataset_counts = {"mesa": dict(stage_counts), "stages": dict(stage_counts)}

for _, row in included.iterrows():
    subj_id = row["subject_id"]
    dataset = row["dataset"]
    ds_dir = DATASETS_DIR / dataset
    annot_files = list((ds_dir / "polysomnography" / "annotations-events-nsrr").glob(f"*{subj_id}*.xml"))
    
    if not annot_files:
        continue
    
    _, stages = load_hypnogram(annot_files[0])
    for stage_val, stage_name in stage_map_inv.items():
        count = int(np.sum(stages == stage_val))
        stage_counts[stage_name] += count
        per_dataset_counts[dataset][stage_name] += count

print("Overall label distribution (30-sec epochs):")
total = sum(stage_counts.values())
for name, count in stage_counts.items():
    pct = count / total * 100 if total > 0 else 0
    print(f"  {name}: {count:>8,} ({pct:.1f}%)")
print(f"  Total: {total:>8,}")

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ["#e74c3c", "#f39c12", "#3498db", "#2c3e50", "#2ecc71"]
class_names = ["WAKE", "N1", "N2", "N3", "REM"]

# Overall
values = [stage_counts[n] for n in class_names]
axes[0].bar(class_names, values, color=colors)
axes[0].set_title("Overall Label Distribution")
axes[0].set_ylabel("Epoch Count")

# Per dataset
for i, dataset in enumerate(["mesa", "stages"]):
    vals = [per_dataset_counts[dataset][n] for n in class_names]
    axes[i+1].bar(class_names, vals, color=colors)
    axes[i+1].set_title(f"{dataset.upper()} Label Distribution")
    axes[i+1].set_ylabel("Epoch Count")

plt.tight_layout()
plt.show()

# Class imbalance ratio
print("\nClass imbalance ratios (relative to minority class):")
min_count = min(v for v in stage_counts.values() if v > 0)
for name, count in stage_counts.items():
    ratio = count / min_count if min_count > 0 else 0
    print(f"  {name}: {ratio:.1f}x")
print("\n→ Use class_weights='balanced' in training config to handle imbalance.")

## 5. Recording Duration Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i, dataset in enumerate(["mesa", "stages"]):
    subset = included[included["dataset"] == dataset]
    axes[i].hist(subset["psg_duration_hours"], bins=30, color="#3498db", edgecolor="white")
    axes[i].axvline(x=subset["psg_duration_hours"].median(), color="red", linestyle="--",
                    label=f'Median: {subset["psg_duration_hours"].median():.1f}h')
    axes[i].set_title(f"{dataset.upper()} PSG Duration")
    axes[i].set_xlabel("Hours")
    axes[i].set_ylabel("Count")
    axes[i].legend()

plt.tight_layout()
plt.show()

## Summary

Key findings from this exploration:
- Number of valid subjects per dataset
- Class imbalance (N1 is typically the most under-represented)
- Signal availability (Pleth, SpO2, HR coverage)
- Recording quality distribution

**Next steps**: Proceed to feature extraction (Phase 2) using the validated subject list.